# OAKG — FLARE evaluation (all FLARE analyses in one place)

How the two FLARE **data sources** are used in OAKG, each at its precise evaluation level.
Runs on the **committed artifacts** (`benchmark/`, `results/`) — no raw masks needed.

- **FLARE22** — organs, patient-level GT (via the FLARE 2024–2025 Task2 release): the main multi-organ hub.
- **FLARE23** — class-14 tumor, real GT but **slice-level** (patient identity lost): exploratory only.

**Sections:** 1. Roles & provenance · 2. Valid cancer predicates per dataset · 3. FLARE22 UNOBSERVED-tumor observability · 
4. FLARE22's benchmark contribution · 5. Cancer-query retrievals (MSD/LiTS/FLARE22) · 6. FLARE23 slice-level tumor — why exploratory + retrievals


In [1]:
import json, pandas as pd, numpy as np
from pathlib import Path
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 70)
ROOT = Path.cwd()
while not ((ROOT/'benchmark').exists() and (ROOT/'results').exists()) and ROOT != ROOT.parent:
    ROOT = ROOT.parent
B, R = ROOT/'benchmark', ROOT/'results'
assert (B/'case_scopes.csv').exists(); print('committed artifacts found (benchmark/, results/)')


committed artifacts found (benchmark/, results/)


## 1. FLARE dataset roles & provenance

Full provenance in `results/audit/flare_provenance.md`. Patient-ID column answers whether
independent-query significance is possible.


In [2]:
roles = pd.DataFrame([
  ['FLARE22','5 organs; no tumor annotations','patient (3D vol)','YES','tests UNOBSERVED tumor; cross-source observability','3D organs (if NIfTI avail)'],
  ['FLARE23 (local)','class-14 tumor','slice (2D)','NO (lost)','EXPLORATORY tumor presence/area/organ-overlap','2D only'],
  ['MSD Pancreas','pancreas + tumor','patient (3D vol)','YES','pancreatic tumor presence/burden/multiplicity/containment','3D'],
  ['LiTS','liver + tumor','patient (3D vol)','YES','liver tumor presence/burden/multiplicity/containment','3D'],
], columns=['dataset','labels','eval level','patient ID preserved','cancer-query use','visualization'])
roles


,dataset,labels,eval level,patient ID preserved,cancer-query use,visualization
0,FLARE22,5 organs; no tumor annotations,patient (3D vol),YES,tests UNOBSERVED tumor; cross-source observability,3D organs (if NIfTI avail)
1,FLARE23 (local),class-14 tumor,slice (2D),NO (lost),EXPLORATORY tumor presence/area/organ-overlap,2D only
2,MSD Pancreas,pancreas + tumor,patient (3D vol),YES,pancreatic tumor presence/burden/multiplicity/containment,3D
3,LiTS,liver + tumor,patient (3D vol),YES,liver tumor presence/burden/multiplicity/containment,3D


## 2. Valid cancer predicates per dataset

Which tumor phenotypes each source can actually answer — derived from the annotation-capability
scope (`annotation_scopes.json`). FLARE22 supports **no** tumor predicate (tumor is UNOBSERVED).


In [3]:
ann = json.load(open(B/'annotation_scopes.json'))['annotation_capability_scopes']
GLOBAL=['has_tumor','tumor_burden_cm3','lesion_multiplicity']
rows=[]
for src,label in [('msd_pancreas','MSD Pancreas'),('lits','LiTS'),('flare22','FLARE22')]:
    t=ann[src]['annotates_tumor_for']
    preds=[]
    for o in t: preds += [f'{o}_tumor_present', f'{o}_tumor_containment']
    if t: preds += GLOBAL
    rows.append([label, ', '.join(preds) if preds else 'NONE (tumor UNOBSERVED)'])
rows.append(['FLARE23 (slice)','slice tumor_present, tumor_area, cross_organ_tumor (SLICE-LEVEL only)'])
pd.DataFrame(rows, columns=['dataset','valid cancer predicates'])


,dataset,valid cancer predicates
0,MSD Pancreas,"pancreas_tumor_present, pancreas_tumor_containment, has_tumor, tum..."
1,LiTS,"liver_tumor_present, liver_tumor_containment, has_tumor, tumor_bur..."
2,FLARE22,NONE (tumor UNOBSERVED)
3,FLARE23 (slice),"slice tumor_present, tumor_area, cross_organ_tumor (SLICE-LEVEL only)"


## 3. FLARE22 observability — pancreatic tumor stays UNOBSERVED

A FLARE22 case observes the pancreas **organ** but its source annotates **no tumor**, so
`pancreas_tumor_present` is **Unknown** (not tumor-negative). Computed from anatomy ∩ annotation-capability.


In [4]:
scopes = pd.read_csv(B/'case_scopes.csv'); units = json.load(open(B/'support_units.json'))['support_units']
def observe(row, feature):
    organs=set(str(row.observed_organs).split('|')); tannot=set(ann[row.source_id]['annotates_tumor_for']); need=units[feature]
    anat=all(o in organs for o in need['anatomy'])
    cap=all((u=='tumor_annotation:*' and len(tannot)>0) or (u.startswith('tumor_annotation:') and u.split(':')[1] in tannot) for u in need['annotation_capability'])
    return 'OBSERVED (T/F)' if (anat and cap) else 'UNOBSERVED (U)'
msd=scopes[scopes.source_id=='msd_pancreas'].iloc[0]; fl=scopes[scopes.source_id=='flare22'].iloc[0]
print(f'MSD case {msd.case_id}  vs  FLARE case {fl.case_id}')
pd.DataFrame([[f,observe(msd,f),observe(fl,f)] for f in ['pancreas_present','pancreas_volume_cm3','pancreas_tumor_present']],
             columns=['phenotype','MSD-Pancreas','FLARE22'])


MSD case pancreas_001  vs  FLARE case FLARE22_Tr_0001


,phenotype,MSD-Pancreas,FLARE22
0,pancreas_present,OBSERVED (T/F),OBSERVED (T/F)
1,pancreas_volume_cm3,OBSERVED (T/F),OBSERVED (T/F)
2,pancreas_tumor_present,OBSERVED (T/F),UNOBSERVED (U)


**Key:** in an MSD↔FLARE pair `pancreas_tumor_present` is *tumor-incomparable* — excluded, never scored tumor-negative.


## 4. FLARE22's contribution to the main patient-level benchmark

The multi-organ hub — without it γ is degenerate and the policy ablation, native cross-dataset,
and observation-boundary results are trivial or impossible.


In [5]:
print('Policy ablation:'); print(pd.read_csv(R/'tables'/'ranking_policy_selection.csv').to_string(index=False))
nm=pd.read_csv(R/'native_cross_dataset'/'native_method_summary.csv')
print('\nNative cross-dataset, FLARE22 as bridge (flare22 -> others):')
print(nm[(nm.stratum=='flare22-to-other-sources')&(nm.method.isin(['OAKG','OAKG-Union']))][['stratum','method','mean','n_queries']].to_string(index=False))
print('\nObservation-boundary ablation:')
print(pd.read_csv(R/'union_ablation'/'union_ablation_paired_comparison.csv')[['regime','delta_obs','ci_low','ci_high','holm_p']].to_string(index=False))


Policy ablation:
                  method  mean_metric  served_rate  n_queries
OAKG-lexicographic [ref]     0.402694          1.0        111
      OAKG-product [ref]     0.402694          1.0        111
   OAKG-similarity [ref]     0.317940          1.0        111
    OAKG-threshold [ref]     0.316318          1.0        111

Native cross-dataset, FLARE22 as bridge (flare22 -> others):
                 stratum     method     mean  n_queries
flare22-to-other-sources       OAKG 0.179272          8
flare22-to-other-sources OAKG-Union 0.000000          8

Observation-boundary ablation:
       regime  delta_obs    ci_low  ci_high  holm_p
   asymmetric   0.068095  0.022390 0.114561  0.0004
dataset_style   0.111879  0.044522 0.182173  0.0004
       random  -0.009730 -0.023495 0.000261  0.0004
      uniform   0.000769  0.000000 0.002307  1.0000


## 5. Cancer-query retrievals (patient-level, MSD / LiTS / FLARE22)

Actual OAKG top-5 retrievals for tumor queries. ✓ = all query conditions matched (relevant).


In [6]:
qr=pd.read_csv(R/'qualitative'/'qualitative_retrieval.csv')
tumor_qs=[q for q in qr.query_id.unique() if 'tum' in qr[qr.query_id==q].need.iloc[0].lower()]
for q in tumor_qs[:3]:
    g=qr[qr.query_id==q].sort_values('rank'); need=g.need.iloc[0]; n=int(g.n_conditions.iloc[0])
    print(f'\n{q}: {need}  (anchor {g.anchor.iloc[0]}, {g.dataset.iloc[0]})')
    for r in g.itertuples():
        mark='✓' if r.conditions_matched==n else f'{r.conditions_matched}/{n}'
        print(f'   {r.rank}. {r.candidate:16s} [{mark}]  {r.query_feature_values}')



Q1: Pancreatic tumour, contained, in a large pancreas  (anchor pancreas_004, Pancreas)
   1. pancreas_124     [✓]  pancreas_tumor_present=1.0; pancreas_tumor_containment=1.0; pancreas_volume_cm3=102.0
   2. pancreas_070     [✓]  pancreas_tumor_present=1.0; pancreas_tumor_containment=1.0; pancreas_volume_cm3=93.7
   3. pancreas_364     [✓]  pancreas_tumor_present=1.0; pancreas_tumor_containment=1.0; pancreas_volume_cm3=95.0
   4. pancreas_293     [✓]  pancreas_tumor_present=1.0; pancreas_tumor_containment=1.0; pancreas_volume_cm3=98.9
   5. pancreas_102     [✓]  pancreas_tumor_present=1.0; pancreas_tumor_containment=1.0; pancreas_volume_cm3=99.5

Q2: Liver tumour, multifocal, high tumour burden  (anchor LiTs-004, LiTS)
   1. LiTs-100         [✓]  liver_tumor_present=1.0; lesion_multiplicity=11.0; tumor_burden_cm3=395.5
   2. LiTs-130         [✓]  liver_tumor_present=1.0; lesion_multiplicity=23.0; tumor_burden_cm3=346.2
   3. LiTs-116         [2/3]  liver_tumor_present=1.0; lesion_multi

## 6. FLARE23 slice-level tumor stratum — EXPLORATORY, not confirmatory

**Why exploratory, not confirmatory:**
1. The **364 slices are not 364 independent samples** — CIs/p-values assume independence, but adjacent
   slices of the same patient are near-duplicate anatomy (highly correlated), so the effective N is far smaller.
2. **Patient clustering is unrecoverable** — the class-stack format lost patient identity, so we cannot
   resample by patient (cluster-robust bootstrap) to get honest CIs.
3. Treating slices as independent would make CIs too narrow / p-values anti-conservative (falsely significant),
   so we report only the **directional** paired delta and make **no** patient-level significance claim.
4. It is slice-level 2D (tumor *areas*), not patient-level 3D (*volumes*) — a different granularity, kept separate.

So: a *directional real-label corroboration* that the OAKG−imputation separation also appears on real tumor
data — **not** a confirmatory patient-level result.


In [7]:
print('FLARE23 cross-organ tumor stratum (method summary, EXPLORATORY, paired deltas only):')
print(pd.read_csv(R/'strata'/'flare_tumor_realgt.csv').to_string(index=False))
print('\nExample slice-level retrievals (OAKG top-5 for cross-organ-tumor slice queries):')
ex=pd.read_csv(R/'strata'/'flare23_tumor_retrieval_examples.csv')
for q in ex.query_id.unique():
    g=ex[ex.query_id==q]
    print(f'  query slice {g.query_slice.iloc[0]}: top-5 relevant = {int(g.relevant.sum())}/5 '
          f'(all cross-organ tumor: {int(g.cross_organ_tumor.sum())}/5)')
print('\n(Directional only — these slices cannot be treated as independent patients.)')


FLARE23 cross-organ tumor stratum (method summary, EXPLORATORY, paired deltas only):
   method  nDCG@10  delta_vs_ZeroImp  ci_low  ci_high  served_rate sig
       WL   0.6407            0.0677  0.0412   0.0945          1.0 SIG
     OAKG   0.6157            0.0427  0.0194   0.0668          1.0 SIG
  ZeroImp   0.5730            0.0000  0.0000   0.0000          1.0 NaN
  MissInd   0.5728           -0.0002 -0.0010   0.0005          1.0  ns
  MeanImp   0.5147           -0.0582 -0.0822  -0.0341          1.0 SIG
MaskedCos   0.1912           -0.3817 -0.4204  -0.3424          1.0 SIG

Example slice-level retrievals (OAKG top-5 for cross-organ-tumor slice queries):
  query slice FLARE-9c96a6cfb5: top-5 relevant = 5/5 (all cross-organ tumor: 5/5)
  query slice FLARE-804c5f6fa0: top-5 relevant = 5/5 (all cross-organ tumor: 5/5)
  query slice FLARE-6fa5985097: top-5 relevant = 5/5 (all cross-organ tumor: 5/5)

(Directional only — these slices cannot be treated as independent patients.)


---
*Provenance: `results/audit/flare_provenance.md`; observability bridge: `results/audit/annotation_capability.md`.*
